이 노트북을 실행하는 데 필요한 라이브러리(표준 라이브러리 제외)
- torch
- numpy
- matplotlib
- transformers

- 실습 기본 환경 설정


In [1]:

# 예제 실행 및 시각화를 위한 공통 라이브러리 로딩

# 코랩 환경 등 깃허브 전체를 clone해서 실습하는 경우가 아니라면 
# 공통 라이브러리를 불러오기 위해서 별도의 과정이 필요하므로 code_reference/README.md 파일을 확인하자.
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

# 시각화 결과를 파일에 저장하지 않음
viz.configure(save_grayscale=False)

# matplotlib 시각화에서 한글 폰트 사용 설정
common.set_korean_plot_env()

# 재현성 보장을 위한 시드 고정
#   여기서 재현성은 '동일 컴퓨터, 동일 버전의 파이썬, 동일 버전의 파이토치' 환경에서 재현이 가능하다는 의미로, 
#   독자의 결과는 저자의 결과와 달라질 수 있다는 점을 밝혀둔다.
#   또한 같은 환경에서도 GPU를 사용하는 경우, 일부 연산의 비결정적 성질로 인해 실행시마다 결과가 조금씩 달라질 수 있다.
SEED = 42
common.set_seed(SEED)

# 실습 환경에 맞는 하드웨어 가속기 장치 객체
device = common.get_device()

CUDA를 사용합니다.


# 12-3 양자화로 LLM 가볍게 돌리기

본 노트북은 본문 12-3절의 코드 예제와 관련 내용을 다룬다. 주요 내용은 다음과 같다.
- `BitsAndBytesConfig`로 만드는 양자화 설정 객체와 4비트 양자화 모델 로드
- 양자화 전후의 메모리 사용량과 답변 품질 비교([표 12-9])
- (참고) QLoRA로 함수 호출 작업에 미세 조정하는 전체 과정

> 이번 절 예제는 NVIDIA GPU 환경에서 실습하기를 권장한다. NVIDIA GPU를 사용할 수 없다면 구글 코랩의 무료 티어도 좋은 선택지다.

## 답변 생성 헬퍼

- 12-1절 [코드 12-7]을 모델과 토크나이저를 인자로 받는 함수로 묶었다.
    - 양자화 모델과 비양자화 모델에 같은 방식으로 질문하기 위해서다.

In [2]:
# 참고 - 답변 생성 헬퍼 (12-1절 [코드 12-7]을 함수로 묶은 것)

import torch

def generate_with_llama(prompt, model, tokenizer, max_new_tokens=256):
    # LLaMA 계열은 종료 토큰이 두 가지다 (<|end_of_text|>, <|eot_id|>).
    terminators = [
        tokenizer.convert_tokens_to_ids('<|end_of_text|>'),
        tokenizer.convert_tokens_to_ids('<|eot_id|>'),
    ]
    messages = [
        {'role': 'system',
         'content': '당신은 한국어를 사용하는 친절한 AI 친구입니다.'},
        {'role': 'user', 'content': prompt},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors='pt', return_dict=False,
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            eos_token_id=terminators,
            do_sample=True, temperature=0.6, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    llm_generated = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(llm_generated, skip_special_tokens=True)


def gpu_memory_mb():
    """현재 PyTorch가 GPU에 할당한 메모리를 MB 단위로 반환."""
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.memory_allocated() / 1024 / 1024

## 4비트 양자화로 모델 불러오기

- bitsandbytes 양자화는 모델을 불러들인 후 따로 적용하는 것이 아니라, 불러오는 단계에서 함께 적용한다.
    - [코드 12-2]의 모델 로드에 양자화 설정 객체 인자(`quantization_config=bnb_config`)만 추가하면 된다.
- 본문 [표 12-7]은 대표적인 PTQ 방식을, [표 12-8]은 모델 파라미터 수별 양자화 추론 메모리 추정치를 정리한다.

In [3]:
######################################################################################
# 코드 12-13 - 양자화 설정 객체를 만들고 4비트 양자화로 모델 불러오기
######################################################################################

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = 'Bllossom/llama-3.2-Korean-Bllossom-3B'

# 4비트 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # 4비트로 모델 가중치 로드
    bnb_4bit_quant_type='nf4',              # NF4 형식 사용
    bnb_4bit_compute_dtype=torch.float16,   # 행렬곱 계산은 FP16으로
    bnb_4bit_use_double_quant=True,         # 양자화 상수도 한 번 더 양자화
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_q = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map='auto',                      # 사용 가능한 장치에 자동 배치
)

# 양자화 모델을 불러온 직후의 가속기 메모리를 측정해 두었다가 뒤에서 비교한다
q_memory_mb = gpu_memory_mb()
print(f'양자화 모델 로딩 후 GPU 메모리: {q_memory_mb:.0f} MB')

/home/crapas/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/254 [00:00<01:35,  2.65it/s]

Loading weights:   1%|          | 3/254 [00:00<00:35,  7.11it/s]

Loading weights:   5%|▌         | 13/254 [00:00<00:07, 31.01it/s]

...

Loading weights:  91%|█████████ | 231/254 [00:03<00:00, 65.92it/s]

Loading weights:  94%|█████████▍| 239/254 [00:03<00:00, 64.85it/s]

Loading weights:  98%|█████████▊| 248/254 [00:04<00:00, 66.45it/s]

Loading weights: 100%|██████████| 254/254 [00:04<00:00, 61.94it/s]

양자화 모델 로딩 후 GPU 메모리: 2139 MB


- `torch.cuda.memory_allocated()`로 메모리를 비교하고, 양자화하지 않은 FP16 모델과 답변 품질도 함께 본다([표 12-9]).
    - 호출할 때마다 다른 답변을 생성하므로, 여러 번 호출해 경향을 확인하는 편이 좋다.

In [4]:
# 참고 - 양자화(NF4)/비양자화(FP16) 모델의 메모리와 답변 비교
import gc

prompt = '대규모 언어 모델을 적은 자원으로 다루는 방법을 알려 줘.'

print('--- 양자화 적용(NF4) 모델 답변 ---')
print(generate_with_llama(prompt, model_q, tokenizer))

# FP16 비교군 적재를 위해 양자화 모델을 잠시 비운다
del model_q
gc.collect()
torch.cuda.empty_cache()

# 양자화 없이 불러온 비교군 (FP16)
model_fp = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16,
).to(device)
fp_memory_mb = gpu_memory_mb()

print(f'\n양자화 모델 메모리   : {q_memory_mb:.0f} MB')
print(f'비양자화(FP16) 메모리: {fp_memory_mb:.0f} MB')
print('\n--- 양자화 미적용(FP16) 모델 답변 ---')
print(generate_with_llama(prompt, model_fp, tokenizer))

# 비교가 끝나면 FP16 모델은 해제한다 (QLoRA 학습용 메모리 확보)
del model_fp
gc.collect()
torch.cuda.empty_cache()
print(f'\n해제 후 GPU 메모리: {gpu_memory_mb():.0f} MB')

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


--- 양자화 적용(NF4) 모델 답변 ---


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


대규모 언어 모델을 적은 자원으로 다루는 방법은 여러 가지로 나눌 수 있습니다. 아래는 몇 가지 방법을 소개합니다.

1. **GPU(그래픽 프로세서) 사용**: 대규모 언어 모델은 많은 컴퓨팅 자원을 필요로합니다. GPU를 사용하여 모델의 컴퓨팅 자원을 효율적으로 사용할 수 있습니다. NVIDIA의 CUDA 또는 AMD의 ROCm와 같은 GPU 소프트웨어를 사용하여 모델의 성능을 향상시킬 수 있습니다.

2. **Cloud 서비스 사용**: Cloud 서비스인 AWS, Google Cloud, Microsoft Azure, etc.를 사용하여 대규모 언어 모델을 실행할 수 있습니다. 이 서비스는 인프라 관리, 스케일링, 보안, etc.를 제공합니다.

3. **Containerization**: Docker와 같은 컨테이너 소프트웨어를 사용하여 모델의 컴퓨팅 자원을 효율적으로 관리할 수 있습니다. 컨테이너를 사용하면 모델의 컴퓨팅 자원을 효율적으로 사용할 수 있으며, 또한 보안과 관리를 쉽게 할 수 있습니다.

4. **Distributed Computing**: 여러 기계를 사용하여 모델의 컴퓨팅 자원을 분


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

Loading weights:  99%|█████████▉| 251/254 [00:02<00:00, 91.77it/s] 

Loading weights: 100%|██████████| 254/254 [00:02<00:00, 93.78it/s]


양자화 모델 메모리   : 2139 MB
비양자화(FP16) 메모리: 6136 MB

--- 양자화 미적용(FP16) 모델 답변 ---


대규모 언어 모델을 적은 자원으로 다루는 방법은 여러 가지가 있습니다. 아래는 그 중 일부입니다:

1. **모델 조각**: 대규모 언어 모델은 그 크기가 매우 큽니다. 이를 위해 모델을 여러 부분으로 나누어 관리할 수 있습니다. 이를 통해 각 부분이 적은 자원을 사용할 수 있습니다.

2. **다중 서버**: 여러 서버를 사용하여 모델을 분산하여 처리할 수 있습니다. 각 서버는 모델의 일부를 처리할 수 있으며, 이를 통해 시스템의 overall 처리 속도를 높일 수 있습니다.

3. **클라우드 서비스**: 클라우드 서비스를 이용하여 대규모 언어 모델을 처리할 수 있습니다. 예를 들어, AWS SageMaker, Google Cloud AI Platform, Microsoft Azure Machine Learning 등이 있습니다.

4. **모델 추적**: 모델의 성능을 지속적으로 추적하여 적절한 조정을 할 수 있습니다. 이는 모델의 효율성을 높이고, 자원을 최적화하는 데 도움을 줍니다.

5. **모델 최적화**: 모델의 크기와 복잡성을 최소화하여 자원을 절약할 수 있습니다. 이를



해제 후 GPU 메모리: 8 MB


## 참고 - QLoRA로 미세 조정하기

- 양자화로 큰 모델을 내 환경에 맞게 사용할 수 있다면, 미세 조정도 가능하지 않을까?
    - PEFT의 대표 기법인 LoRA에 양자화를 적용한 QLoRA가 이에 해당하는 양자화 미세 조정 방식이다.
- 최종 원고에서는 QLoRA 실습이 본문에서 빠지고 심화 학습 자료로 옮겨졌다.
    - LoRA와 QLoRA 기법 소개, 메모리 예산 계산, 그리고 Bllossom-3B 모델을 4비트 양자화해 미세 조정하는 실습 과정을 저장소의 `articles/qlora/`에 정리해 두었다.
    - 실습 코드는 [`articles/qlora/qlora_example.ipynb`](../../articles/qlora/qlora_example.ipynb)에 있다.
- 입문 단계에서는 당장 필요하지 않더라도, 실무에서 큰 모델을 다루게 되면 반드시 마주치는 기법이다.

## 정리

- 양자화는 모델을 불러오는 단계에서 함께 적용한다. `Auto` 모델 클래스에 양자화 설정 객체만 넘기면 된다.
- 4비트 양자화는 메모리를 크게 줄이지만 품질 손실이 따른다. 정밀한 답변이 필요한 작업에서는 8비트를 쓰거나 양자화하지 않는 편이 안전하다.
- LoRA 어댑터는 사전 학습 파라미터를 고정한 채 작은 행렬만 학습하므로, 양자화 모델에도 적용할 수 있다(QLoRA).